# Simulating the Linear Space-Time QLBM

The space-time method encodes the light cone of every gridpoint, so the register holds one qubit
per (gridpoint, neighbour, velocity) triple that can influence the result within `num_timesteps`
steps. Streaming then becomes a permutation of that neighbourhood.

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

A space-time circuit is only valid for the `num_timesteps` it was built with. The
`SpaceTimeReinitializer` therefore re-synthesizes the neighbourhood encoding from the sampled
populations at the end of every time step — it consumes counts, not a statevector copy. Only the
snapshot time loop consults the reinitializer, so `statevector_snapshots=True` is a correctness
requirement here rather than only a speed-up.

> **Demo-sized.** Register width grows fast: a D2Q4 8x8 lattice with `num_timesteps=1`
> needs 26 qubits (a 1 GB statevector), and `num_timesteps=2` needs 58. The D1Q2 lattice below
> uses 14 and runs instantly. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import EmptyPrimitive
from qlbm.components.spacetime import (
    PointWiseSpaceTimeInitialConditions,
    SpaceTimeGridVelocityMeasurement,
    SpaceTimeQLBM,
)
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import SpaceTimeLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = SpaceTimeLattice(
    num_timesteps=2,
    lattice_data={
        "lattice": {"dim": {"x": 16}, "velocities": "D1Q2"},
        "geometry": [],
    },
    use_volumetric_ops=False,
)

output_dir = "qlbm-output/spacetime-d1q2-16-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**10

# Number of time steps to simulate
NUM_STEPS = 10

In [ ]:
cfg = SimulationConfig(
    initial_conditions=PointWiseSpaceTimeInitialConditions(
        lattice, grid_data=[((4,), (True, True)), ((5,), (True, True))]
    ),
    algorithm=SpaceTimeQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=SpaceTimeGridVelocityMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
# Create a runner object to simulate the circuit
runner = QarpRunner(cfg, lattice, seed=1234)

In [ ]:
# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS).
# The runner reuses one command stream for every step; the reinitializer
# supplies each step's initial state.
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)